# 03 — Modeling and Backtest: Does the Signal + Regime Model Actually Work?

This notebook runs and visualises the walk-forward backtest of
`RegimePredictor` (logistic regression on momentum/RSI/trend signals,
conditioned on macro regime) against a buy-and-hold ALSI baseline.

The headline result, established in `scripts/run_backtest.py`, is a
**negative finding**: across 111 monthly rebalance periods, the model's
top-5 picks beat the index only 49.5% of the time — statistically
indistinguishable from chance. This notebook exists to show that result
properly, examine it from several angles, and draw an honest conclusion
rather than either overselling or dismissing it.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import sys

sys.path.insert(0, str(Path("..").resolve() / "src"))

from jse_radar.config import PROC_MASTER_DIR
from jse_radar.modeling.regime_predictor import build_outperformance_label, RegimePredictor
from jse_radar.modeling.backtester import run_walk_forward_backtest, summarise_backtest

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

Imports OK


In [2]:
master_files = sorted(PROC_MASTER_DIR.glob("master_regimes_*.parquet"))
df = pd.read_parquet(master_files[-1])
df["date"] = pd.to_datetime(df["date"])

df = build_outperformance_label(df)

print("Running walk-forward backtest (refits at every monthly rebalance date — this takes a minute or two)...")
results = run_walk_forward_backtest(df)
summary = summarise_backtest(results)

print(f"\nPeriods evaluated: {summary['n_periods']}")
print(f"Hit rate: {summary['hit_rate']:.1%}")
print(f"Avg net excess return per period: {summary['avg_excess_return_net']:.4f}")
print(f"Sharpe-style ratio: {summary['sharpe_style_ratio']}")
print(f"\n{summary['note']}")

2026-06-30 10:01:16 | INFO     | jse_radar.modeling.regime_predictor | Dropped 6,966 rows with missing features/label


Running walk-forward backtest (refits at every monthly rebalance date — this takes a minute or two)...


2026-06-30 10:01:16 | WARNING  | jse_radar.modeling.backtester | Skipping 2016-01-01 — fit failed: Found array with 0 sample(s) (shape=(0, 3)) while a minimum of 1 is required by StandardScaler.
2026-06-30 10:01:16 | INFO     | jse_radar.modeling.regime_predictor | Dropped 7,506 rows with missing features/label
2026-06-30 10:01:16 | WARNING  | jse_radar.modeling.backtester | Skipping 2016-02-01 — fit failed: Found array with 0 sample(s) (shape=(0, 3)) while a minimum of 1 is required by StandardScaler.
2026-06-30 10:01:16 | INFO     | jse_radar.modeling.regime_predictor | Dropped 7,506 rows with missing features/label
2026-06-30 10:01:17 | INFO     | jse_radar.modeling.regime_predictor | Fitted RegimePredictor on 567 rows, 1 regime categories, base rate (outperformed=1): 0.466
2026-06-30 10:01:17 | INFO     | jse_radar.modeling.regime_predictor | Dropped 7,506 rows with missing features/label
2026-06-30 10:01:17 | INFO     | jse_radar.modeling.regime_predictor | Fitted RegimePredictor 


Periods evaluated: 111
Hit rate: 49.5%
Avg net excess return per period: 0.0030
Sharpe-style ratio: 0.21

Evaluated over 111 monthly rebalance periods. Reasonable sample size for a directional conclusion, though still a single historical path, not a guarantee.


## 1. Cumulative performance: model picks vs holding the ALSI

The single most intuitive chart for any backtest: if you had actually
followed the model's picks every month and compounded the (net of cost)
excess return, where would you be relative to simply holding the index?
A line that ends near zero or drifts below it is the visual confirmation
of a coin-flip-or-worse result.

In [3]:
results_sorted = results.sort_values("rebalance_date").reset_index(drop=True)
results_sorted["cumulative_net_excess"] = (
    (1 + results_sorted["net_excess_return"]).cumprod() - 1
)
results_sorted["cumulative_gross_excess"] = (
    (1 + results_sorted["excess_return"]).cumprod() - 1
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=results_sorted["rebalance_date"], y=results_sorted["cumulative_gross_excess"] * 100,
    name="Cumulative excess (before costs)", line={"color": "#58a6ff", "width": 1.5, "dash": "dot"},
))
fig.add_trace(go.Scatter(
    x=results_sorted["rebalance_date"], y=results_sorted["cumulative_net_excess"] * 100,
    name="Cumulative excess (after costs)", line={"color": "#f0b429", "width": 2},
))
fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.4,
              annotation_text="Equal to holding the ALSI")

fig.update_layout(
    title="Cumulative Excess Return: Model Picks vs Holding the ALSI<br>"
          "<sup>A line ending near or below zero means the model added no real value</sup>",
    yaxis_title="Cumulative excess return (%)",
    template="plotly_dark",
    height=500,
    hovermode="x unified",
)
fig.show()

print(f"Final cumulative excess (net of costs): {results_sorted['cumulative_net_excess'].iloc[-1]:.1%}")

Final cumulative excess (net of costs): 21.5%


## 2. Hit rate over time — is there any period where it worked?

An overall 49.5% hit rate could in principle be hiding a real edge that
existed in one era and vanished in another (e.g. it might have worked
during a specific regime and failed elsewhere). We check this with a
rolling hit rate to see if there's any genuinely strong stretch, or if
the coin-flip result is consistent throughout.

In [4]:
results_sorted["rolling_hit_rate_12m"] = (
    results_sorted["beat_alsi"].rolling(window=12, min_periods=6).mean()
)

fig = px.line(
    results_sorted,
    x="rebalance_date", y="rolling_hit_rate_12m",
    title="12-Month Rolling Hit Rate<br>"
          "<sup>0.50 = coin flip. Sustained periods above ~0.55 would suggest a real, if modest, edge</sup>",
    labels={"rolling_hit_rate_12m": "Rolling hit rate", "rebalance_date": "Date"},
    template="plotly_dark",
)
fig.add_hline(y=0.5, line_dash="dash", line_color="white", opacity=0.5,
              annotation_text="Coin flip")
fig.update_layout(height=450, yaxis_range=[0, 1])
fig.show()

## 3. Which feature actually mattered? Coefficient inspection

Even though the strategy as a whole doesn't show an edge, it's worth
checking the model's own internal logic: did it learn anything
sensible-looking at all, or is it essentially noise? We refit on the
full dataset (not for trading — purely for inspection) and look at
which features the model weighted most heavily.

In [5]:
full_model = RegimePredictor().fit(df.dropna(subset=["outperformed"]))
coefs = full_model.coefficient_summary()

fig = px.bar(
    coefs,
    x="feature", y="coefficient",
    color="coefficient",
    color_continuous_scale="RdYlGn",
    title="Logistic Regression Coefficients — Full-Sample Fit (inspection only, not used for trading)",
    template="plotly_dark",
)
fig.add_hline(y=0, line_dash="dash", opacity=0.4)
fig.update_layout(height=500, xaxis_tickangle=-30, coloraxis_showscale=False)
fig.show()

print(coefs.to_string(index=False))

2026-06-30 10:04:48 | INFO     | jse_radar.modeling.regime_predictor | Dropped 14,315 rows with missing features/label
2026-06-30 10:04:48 | INFO     | jse_radar.modeling.regime_predictor | Fitted RegimePredictor on 64,378 rows, 6 regime categories, base rate (outperformed=1): 0.457


                        feature  coefficient
 regime_HIKING_TARGET_INFLATION       0.0729
    regime_HIKING_LOW_INFLATION      -0.0433
                         rsi_14      -0.0424
   regime_CUTTING_LOW_INFLATION      -0.0419
                   trend_signal      -0.0324
  regime_CUTTING_HIGH_INFLATION      -0.0320
                 momentum_score       0.0300
regime_CUTTING_TARGET_INFLATION      -0.0246
   regime_HIKING_HIGH_INFLATION       0.0007


## 4. Distribution of monthly excess returns

A final honest check: is the average net excess return (+0.30%/month)
being driven by a few lucky outlier months, or is it a genuinely
consistent small edge spread across many periods? A wide, lumpy
distribution with a few large positive outliers and otherwise negative
months is a different (and weaker) story than a tight distribution
centred slightly above zero.

In [6]:
fig = px.histogram(
    results,
    x="net_excess_return",
    nbins=30,
    title="Distribution of Net Excess Return per Rebalance Period",
    labels={"net_excess_return": "Net excess return per period"},
    template="plotly_dark",
)
fig.add_vline(x=0, line_dash="dash", line_color="white", opacity=0.6)
fig.add_vline(
    x=results["net_excess_return"].mean(),
    line_dash="dot", line_color="#f0b429",
    annotation_text=f"mean: {results['net_excess_return'].mean():.3f}",
)
fig.update_layout(height=450)
fig.show()

print(results["net_excess_return"].describe().to_string())

count   109.0000
mean      0.0030
std       0.0489
min      -0.0979
25%      -0.0302
50%      -0.0004
75%       0.0332
max       0.1382


## Conclusion

Across 111 monthly walk-forward rebalance periods spanning roughly 2016
to 2025, a logistic regression using momentum, RSI, and trend-following
signals — conditioned on the macro regime classification built earlier
in this project — produced a **hit rate of 49.5%** against the ALSI
index, with an average net-of-cost excess return of approximately
**0.30% per month** and a Sharpe-style consistency ratio of **0.21**.

**The honest finding is that this does not constitute a demonstrated
edge.** A hit rate this close to 50% is statistically consistent with
the model providing no real information beyond chance, and the modest
average excess return is well within the range that a small number of
favourable months can produce by luck alone — visible directly in the
return distribution above, and in the cumulative excess return chart,
which spends meaningful stretches at or below zero.

**This is not evidence that markets are unpredictable in general, or
that the underlying pipeline is broken.** It is a specific, scoped
finding about a specific, simple model: three technical signals plus a
six-category macro regime label, evaluated honestly with a proper
chronological walk-forward split and realistic rebalancing costs, do
not beat a buy-and-hold index strategy on JSE large-caps over this
period. Three plausible, non-mutually-exclusive explanations:

1. **The features genuinely don't contain enough predictive signal.**
   Momentum and RSI are widely known, heavily arbitraged signals in
   liquid large-cap markets — any edge they once offered may already
   be priced in by other participants.
2. **The sample is still relatively thin.** 111 periods is a reasonable
   number for a directional read, but it is not a large number in
   statistical terms, particularly once regime conditioning fragments
   the data further — the same caution that applied throughout the
   regime analysis notebook applies here too.
3. **A linear model may be too simple to capture genuinely useful
   interactions** between signals and regime — though, per the project's
   own discipline around model complexity vs sample size, a more complex
   model is not automatically the right next step without first
   confirming there's more real signal to extract.

**What this result is genuinely useful for:** it is now a documented,
reproducible baseline. Any future modeling work on this project —
additional features, a different target horizon, a different model
class — has a clear, honestly-measured bar to beat: a 49.5% hit rate
and a 0.21 Sharpe-style ratio. Anything that doesn't clear this baseline,
after the same walk-forward discipline, should not be trusted just
because it "looks better" on a single split.

The infrastructure built to reach this conclusion — the walk-forward
backtester, the chronological split guarantee, the cost-aware evaluation
— is reusable for testing any future hypothesis on this data, regardless
of whether this particular model worked.